In [1]:
import pandas as pd
import glob

In [2]:
files = glob.glob("/Users/mac/Desktop/Upwork/GoldSignalGenerator/Artifacts/MinutelyData/XAU-USD_Minute_*.csv")
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

/var/folders/j6/9x_46crd51ld6f4qmwpg0zgm0000gn/T/ipykernel_45620/2456407113.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


In [3]:
df.shape

(41995, 6)

In [4]:
df["UTC"] = pd.to_datetime(
    df["UTC"], 
    format="%d.%m.%Y %H:%M:%S.%f UTC"
)

df = df.sort_values("UTC", ascending=False).reset_index(drop=True)
df.rename(columns={'UTC':'datetime'}, inplace=True)

In [5]:
print(df.head())

             datetime      Open      High       Low     Close    Volume
0 2025-10-01 12:27:00  3877.775  3879.045  3875.655  3878.925  0.068340
1 2025-10-01 12:26:00  3882.435  3882.465  3877.315  3877.894  0.060030
2 2025-10-01 12:25:00  3880.885  3882.565  3880.175  3882.355  0.049860
3 2025-10-01 12:24:00  3884.954  3884.996  3880.795  3880.885  0.069450
4 2025-10-01 12:23:00  3886.935  3888.705  3884.135  3884.365  0.070477


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41995 entries, 0 to 41994
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   datetime  41995 non-null  datetime64[ns]
 1   Open      41995 non-null  float64       
 2   High      41995 non-null  float64       
 3   Low       41995 non-null  float64       
 4   Close     41995 non-null  float64       
 5   Volume    41995 non-null  float64       
dtypes: datetime64[ns](1), float64(5)
memory usage: 1.9 MB


In [43]:
print("Feature Engineering Started!")
df = df.copy()

Feature Engineering Started!


In [44]:
df.columns

Index(['UTC', 'Open', 'High', 'Low', 'Close', 'Volume', 'SMA_20', 'SMA_50',
       'SMA_200', 'EMA_10', 'EMA_21', 'EMA_50', 'RSI_14', 'RSI_21', 'MACD',
       'MACD_Signal', 'MACD_Histogram', 'BB_Upper', 'BB_Middle', 'BB_Lower'],
      dtype='object')

In [45]:
def calculate_sma(data, window):
    """Simple Moving Average"""
    return data['Close'].rolling(window=window).mean()

def calculate_ema(data, window):
    """Exponential Moving Average"""
    return data['Close'].ewm(span=window).mean()

def calculate_rsi(data, window=14):
    """Relative Strength Index"""
    delta = data['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

def calculate_macd(data, fast=12, sLow=26, signal=9):
    """MACD Indicator"""
    exp1 = data['Close'].ewm(span=fast).mean()
    exp2 = data['Close'].ewm(span=sLow).mean()
    macd = exp1 - exp2
    macd_signal = macd.ewm(span=signal).mean()
    macd_histogram = macd - macd_signal
    return macd, macd_signal, macd_histogram

def calculate_bollinger_bands(data, window=20, num_std=2):
    """Bollinger Bands"""
    sma = data['Close'].rolling(window=window).mean()
    std = data['Close'].rolling(window=window).std()
    upper_band = sma + (std * num_std)
    Lower_band = sma - (std * num_std)
    return upper_band, sma, Lower_band

def calculate_stochastic(data, k_window=14, d_window=3):
    """Stochastic Oscillator"""
    Low_min = data['Low'].rolling(window=k_window).min()
    High_max = data['High'].rolling(window=k_window).max()
    k_percent = 100 * ((data['Close'] - Low_min) / (High_max - Low_min))
    d_percent = k_percent.rolling(window=d_window).mean()
    return k_percent, d_percent

In [46]:
print("Adding Traditional Technical Indicators...")

# Moving Averages 
df['SMA_20'] = calculate_sma(df, 20)    # 20-minute SMA
df['SMA_50'] = calculate_sma(df, 50)    # 50-minute SMA  
df['SMA_200'] = calculate_sma(df, 200)  # 200-minute SMA
df['EMA_10'] = calculate_ema(df, 10)    # 10-minute EMA
df['EMA_21'] = calculate_ema(df, 21)    # 21-minute EMA
df['EMA_50'] = calculate_ema(df, 50)    # 50-minute EMA

# Oscillators
df['RSI_14'] = calculate_rsi(df, 14)
df['RSI_21'] = calculate_rsi(df, 21)

# MACD
df['MACD'], df['MACD_Signal'], df['MACD_Histogram'] = calculate_macd(df)

# Bollinger Bands
df['BB_Upper'], df['BB_Middle'], df['BB_Lower'] = calculate_bollinger_bands(df, 20)

# Stochastic
df['Stoch_K'], df['Stoch_D'] = calculate_stochastic(df)

# ATR (Average True Range)
df['prev_Close'] = df['Close'].shift(1)
df['tr1'] = df['High'] - df['Low']
df['tr2'] = abs(df['High'] - df['prev_Close'])
df['tr3'] = abs(df['Low'] - df['prev_Close'])
df['true_range'] = df[['tr1', 'tr2', 'tr3']].max(axis=1)
df['ATR'] = df['true_range'].rolling(14).mean()
df.drop(['prev_Close', 'tr1', 'tr2', 'tr3'], axis=1, inplace=True)

print("Traditional Indicators added !!!")

Adding Traditional Technical Indicators...
Traditional Indicators added !!!


In [47]:
import numpy as np

print("Started Adding Smart Money Concepts (SMC) Indicators...")

# Break of Structure (BOS) - for 1-minute data, use shorter lookback
df['SMC_BOS_bullish'] = (df['High'] > df['High'].rolling(10).max().shift(1)).astype(int)
df['SMC_BOS_bearish'] = (df['Low'] < df['Low'].rolling(10).min().shift(1)).astype(int)

# Order Blocks - Institutional supply/demand zones
bullish_move = df['Close'] > df['Close'].shift(5)  # 5-minute move
bearish_candle = df['Close'] < df['Open']
df['SMC_bullish_ob'] = (bearish_candle.shift(1) & bullish_move).astype(int)

bearish_move = df['Close'] < df['Close'].shift(5)
bullish_candle = df['Close'] > df['Open']
df['SMC_bearish_ob'] = (bullish_candle.shift(1) & bearish_move).astype(int)

# Fair Value Gaps (FVG) - Price imbalances
df['SMC_bullish_fvg'] = ((df['Low'] > df['High'].shift(2)) & 
                         (df['Close'] > df['Close'].shift(1))).astype(int)
df['SMC_bearish_fvg'] = ((df['High'] < df['Low'].shift(2)) & 
                         (df['Close'] < df['Close'].shift(1))).astype(int)

# Market Structure - Higher Highs, Lower Lows
hh_hl = (df['High'] > df['High'].shift(10)) & (df['Low'] > df['Low'].shift(10))
ll_lh = (df['Low'] < df['Low'].shift(10)) & (df['High'] < df['High'].shift(10))
df['SMC_market_structure'] = np.where(hh_hl, 1, np.where(ll_lh, -1, 0))

# Liquidity Sweeps - Stop hunting
High_sweep = df['High'] > df['High'].rolling(20).max().shift(1)
Low_sweep = df['Low'] < df['Low'].rolling(20).min().shift(1)
df['SMC_liquidity_sweep_High'] = High_sweep.astype(int)
df['SMC_liquidity_sweep_Low'] = Low_sweep.astype(int)

# Change of Character (CHoCH)
df['SMC_choch'] = ((df['SMC_BOS_bullish'] == 1) | (df['SMC_BOS_bearish'] == 1)).astype(int)

print("SMC Indicators Added !!!")

Started Adding Smart Money Concepts (SMC) Indicators...
SMC Indicators Added !!!


In [ ]:
print("Adding Wyckoff Method Indicators...")

# Volume analysis
vol_ma_20 = df['Volume'].rolling(20).mean()
df['Wyckoff_Volume_ratio'] = df['Volume'] / vol_ma_20

# Price Spread Analysis
price_spread = df['High'] - df['Low']
spread_ma = price_spread.rolling(20).mean()

# Effort vs Result
df['Wyckoff_effort_result'] = (df['Volume'] / vol_ma_20) / (price_spread / spread_ma)

# Volume Spike Detection
df['Wyckoff_Volume_spike'] = (df['Volume'] > vol_ma_20 * 2).astype(int)

# High Volume + Narrow Spread (No Demand/Supply)
narrow_spread = price_spread < spread_ma
High_Volume = df['Volume'] > vol_ma_20 * 1.5
df['Wyckoff_High_vol_narrow_spread'] = (High_Volume & narrow_spread).astype(int)

# Accumulation/Distribution Detection
# Range-bound movement with Volume analysis
price_range_std = df['Close'].rolling(30).std()
is_range_bound = price_range_std < price_range_std.rolling(100).mean()

up_movement = df['Close'] > df['Close'].shift(1)
down_movement = df['Close'] < df['Close'].shift(1)

# Accumulation: Range-bound + Up moves on High Volume
df['Wyckoff_accumulation'] = (is_range_bound & up_movement & 
                              (df['Volume'] > vol_ma_20)).astype(int)

# Distribution: Range-bound + Down moves on High Volume  
df['Wyckoff_distribution'] = (is_range_bound & down_movement & 
                              (df['Volume'] > vol_ma_20)).astype(int)

# Volume Price Trend (VPT)
df['Wyckoff_vpt'] = (df['Volume'] * df['Close'].pct_change()).cumsum()
df['Wyckoff_vpt_ma'] = df['Wyckoff_vpt'].rolling(50).mean()
df['Wyckoff_vpt_signal'] = np.where(df['Wyckoff_vpt'] > df['Wyckoff_vpt_ma'], 1, -1)

# Test for Support/Resistance (Low Volume tests)
df['Wyckoff_test_phase'] = (df['Volume'] < vol_ma_20 * 0.7).astype(int)

# Climactic Action (Ultra High Volume)
df['Wyckoff_climax'] = (df['Volume'] > vol_ma_20 * 3).astype(int)
print("Wyckoff Indicators Added !!!")

Adding Wyckoff Method Indicators...
Wyckoff Indicators Added !!!


In [ ]:
print("Adding ICT (Inner Circle Trader) Indicators...")

# Market Maker Buy/Sell Model (MMBM)
# Liquidity raids folLowed by reversals
High_raid = df['High'] > df['High'].rolling(15).max().shift(1)
Low_raid = df['Low'] < df['Low'].rolling(15).min().shift(1)

df['ICT_mmbm_buy_setup'] = (Low_raid.shift(1) & (df['Close'] > df['Open'])).astype(int)
df['ICT_mmbm_sell_setup'] = (High_raid.shift(1) & (df['Close'] < df['Open'])).astype(int)

# Optimal Trade Entry (OTE) - Fibonacci retracement levels
swing_High = df['High'].rolling(20).max()
swing_Low = df['Low'].rolling(20).min()

# Key Fibonacci levels for entries
fib_618 = swing_High - (swing_High - swing_Low) * 0.618
fib_786 = swing_High - (swing_High - swing_Low) * 0.786

df['ICT_ote_buy_zone'] = ((df['Low'] <= fib_786) & (df['High'] >= fib_618)).astype(int)

# Sell OTE zone (inverse)
fib_buy_618 = swing_Low + (swing_High - swing_Low) * 0.618
fib_buy_786 = swing_Low + (swing_High - swing_Low) * 0.786
df['ICT_ote_sell_zone'] = ((df['High'] >= fib_buy_618) & (df['Low'] <= fib_buy_786)).astype(int)

# Displacement - Strong institutional moves
displacement_up = (df['Close'] > df['Close'].shift(5)) & (df['Low'] > df['Low'].shift(5))
displacement_down = (df['Close'] < df['Close'].shift(5)) & (df['High'] < df['High'].shift(5))
df['ICT_displacement'] = np.where(displacement_up, 1, np.where(displacement_down, -1, 0))

# Fair Value Gap (ICT style)
# Same as SMC but with ICT naming convention
df['ICT_fvg_bullish'] = df['SMC_bullish_fvg']
df['ICT_fvg_bearish'] = df['SMC_bearish_fvg']

# Order Block Mitigation
df['ICT_ob_mitigation_bull'] = ((df['Low'] <= df['Low'].shift(10)) & 
                                (df['Close'] > df['Open'])).astype(int)
df['ICT_ob_mitigation_bear'] = ((df['High'] >= df['High'].shift(10)) & 
                                (df['Close'] < df['Open'])).astype(int)

# Market Structure Shift (MSS) - Trend change confirmation
df['ICT_mss_bullish'] = (df['High'] > df['High'].rolling(10).max().shift(1)).astype(int)
df['ICT_mss_bearish'] = (df['Low'] < df['Low'].rolling(10).min().shift(1)).astype(int)

# Liquidity Voids - Gap detection
price_gap = abs(df['Open'] - df['Close'].shift(1))
avg_gap = price_gap.rolling(50).mean()
df['ICT_liquidity_void'] = (price_gap > avg_gap * 2).astype(int)

# Institutional Candles (Large body candles)
candle_body = abs(df['Close'] - df['Open'])
avg_body = candle_body.rolling(50).mean()
df['ICT_institutional_candle'] = (candle_body > avg_body * 2).astype(int)

print("ICT Indicators Added !!!")

Adding ICT (Inner Circle Trader) Indicators...
ICT Indicators Added


In [54]:
df.columns

Index(['UTC', 'Open', 'High', 'Low', 'Close', 'Volume', 'SMA_20', 'SMA_50',
       'SMA_200', 'EMA_10', 'EMA_21', 'EMA_50', 'RSI_14', 'RSI_21', 'MACD',
       'MACD_Signal', 'MACD_Histogram', 'BB_Upper', 'BB_Middle', 'BB_Lower',
       'Stoch_K', 'Stoch_D', 'true_range', 'ATR', 'SMC_BOS_bullish',
       'SMC_BOS_bearish', 'SMC_bullish_ob', 'SMC_bearish_ob',
       'SMC_bullish_fvg', 'SMC_bearish_fvg', 'SMC_market_structure',
       'SMC_liquidity_sweep_High', 'SMC_liquidity_sweep_Low', 'SMC_choch',
       'Wyckoff_Volume_ratio', 'Wyckoff_effort_result', 'Wyckoff_Volume_spike',
       'Wyckoff_High_vol_narrow_spread', 'Wyckoff_accumulation',
       'Wyckoff_distribution', 'Wyckoff_vpt', 'Wyckoff_vpt_ma',
       'Wyckoff_vpt_signal', 'Wyckoff_test_phase', 'Wyckoff_climax',
       'ICT_mmbm_buy_setup', 'ICT_mmbm_sell_setup', 'ICT_ote_buy_zone',
       'ICT_ote_sell_zone', 'ICT_displacement', 'ICT_fvg_bullish',
       'ICT_fvg_bearish', 'ICT_ob_mitigation_bull', 'ICT_ob_mitigation_bear',
 

In [56]:
df.rename(columns={'UTC' : 'datetime'} ,inplace=True)

In [57]:
print("Adding additional ML Features...")

# Price-based features
df['price_change'] = df['Close'].pct_change()
df['price_change_5min'] = df['Close'].pct_change(5)
df['High_Low_ratio'] = df['High'] / df['Low']
df['Close_Open_ratio'] = df['Close'] / df['Open']

# Volatility measures  
df['volatility_10min'] = df['Close'].rolling(10).std()
df['volatility_30min'] = df['Close'].rolling(30).std()

# Volume features
df['Volume_change'] = df['Volume'].pct_change()
df['Volume_ma_ratio'] = df['Volume'] / df['Volume'].rolling(50).mean()

# Time-based features (important for 1-minute data)
df['hour'] = df['datetime'].dt.hour
df['minute'] = df['datetime'].dt.minute
df['weekday'] = df['datetime'].dt.weekday
df['is_market_Open'] = ((df['hour'] >= 9) & (df['hour'] < 17)).astype(int)

# Price position relative to bands
df['price_position_bb'] = (df['Close'] - df['BB_Lower']) / (df['BB_Upper'] - df['BB_Lower'])

# Momentum features
df['momentum_5min'] = df['Close'] / df['Close'].shift(5) - 1
df['momentum_15min'] = df['Close'] / df['Close'].shift(15) - 1

print("Additional ML Features Added !!!")

Adding additional ML Features...
Additional ML Features Added !!!


In [58]:
df.columns

Index(['datetime', 'Open', 'High', 'Low', 'Close', 'Volume', 'SMA_20',
       'SMA_50', 'SMA_200', 'EMA_10', 'EMA_21', 'EMA_50', 'RSI_14', 'RSI_21',
       'MACD', 'MACD_Signal', 'MACD_Histogram', 'BB_Upper', 'BB_Middle',
       'BB_Lower', 'Stoch_K', 'Stoch_D', 'true_range', 'ATR',
       'SMC_BOS_bullish', 'SMC_BOS_bearish', 'SMC_bullish_ob',
       'SMC_bearish_ob', 'SMC_bullish_fvg', 'SMC_bearish_fvg',
       'SMC_market_structure', 'SMC_liquidity_sweep_High',
       'SMC_liquidity_sweep_Low', 'SMC_choch', 'Wyckoff_Volume_ratio',
       'Wyckoff_effort_result', 'Wyckoff_Volume_spike',
       'Wyckoff_High_vol_narrow_spread', 'Wyckoff_accumulation',
       'Wyckoff_distribution', 'Wyckoff_vpt', 'Wyckoff_vpt_ma',
       'Wyckoff_vpt_signal', 'Wyckoff_test_phase', 'Wyckoff_climax',
       'ICT_mmbm_buy_setup', 'ICT_mmbm_sell_setup', 'ICT_ote_buy_zone',
       'ICT_ote_sell_zone', 'ICT_displacement', 'ICT_fvg_bullish',
       'ICT_fvg_bearish', 'ICT_ob_mitigation_bull', 'ICT_ob_mitigat

In [60]:
print("Dataset Shape:", df.shape)
print("\nDataset Info:")
print(df.info())
print("\nFirst few rows:")
print(df.head())

Dataset Shape: (36184, 73)

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36184 entries, 0 to 36183
Data columns (total 73 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   datetime                        36184 non-null  datetime64[ns]
 1   Open                            36184 non-null  float64       
 2   High                            36184 non-null  float64       
 3   Low                             36184 non-null  float64       
 4   Close                           36184 non-null  float64       
 5   Volume                          36184 non-null  float64       
 6   SMA_20                          36165 non-null  float64       
 7   SMA_50                          36135 non-null  float64       
 8   SMA_200                         35985 non-null  float64       
 9   EMA_10                          36184 non-null  float64       
 10  EMA_21                      

In [61]:
# Check for any missing values
print("Missing Values:")
print(df.isnull().sum().sum())
print("\nColumns with missing values:")
missing_cols = df.columns[df.isnull().any()].tolist()
print(missing_cols)

# %%
# Examine price distribution and basic stats
print("Price Statistics (Close):")
print(df['Close'].describe())
print(f"\nPrice Range: {df['Close'].min():.2f} to {df['Close'].max():.2f}")
print(f"Current price change std: {df['price_change'].std():.6f}")

# %%
# Check data frequency and time range
print("Time Range:")
print(f"From: {df['datetime'].min()}")
print(f"To: {df['datetime'].max()}")
print(f"Total records: {len(df)}")

# Calculate time differences to verify 1-minute frequency
time_diff = df['datetime'].diff().dt.total_seconds() / 60
print(f"\nTime frequency check (minutes):")
print(f"Most common interval: {time_diff.mode().iloc[0] if len(time_diff.mode()) > 0 else 'N/A'}")
print(f"Unique intervals: {sorted(time_diff.dropna().unique())[:10]}")  # First 10 unique intervals

# %%
print("Adding Target Columns for Pump Prediction...")

def calculate_pump_targets(df, pump_threshold_percent=0.5):

    df_copy = df.copy()
    
    # Define time horizons (in minutes)
    horizons = [15, 30, 60, 120]
    
    for horizon in horizons:

        future_max = df_copy['High'].shift(-horizon).rolling(window=horizon, min_periods=1).max()
        
        price_increase_pct = ((future_max - df_copy['Close']) / df_copy['Close']) * 100
        
        pump_target = (price_increase_pct >= pump_threshold_percent).astype(int)
        
        df_copy[f'target_pump_{horizon}min'] = pump_target
        df_copy[f'target_pump_{horizon}min_pct'] = price_increase_pct
        
        df_copy[f'future_max_{horizon}min'] = future_max
    
    return df_copy

df_with_targets = calculate_pump_targets(df, pump_threshold_percent=0.5)

print("Target columns added!")
print(f"New dataset shape: {df_with_targets.shape}")

target_cols = [col for col in df_with_targets.columns if col.startswith('target_')]
print("Target Columns:")
for col in target_cols:
    print(f"{col}: {df_with_targets[col].sum()} pumps out of {len(df_with_targets)} records ({df_with_targets[col].mean()*100:.2f}%)")

# %%
# Show sample of the new data with targets
print("Sample data with targets:")
sample_cols = ['datetime', 'Close', 'High', 'Volume'] + target_cols[:4]  # First 4 target columns
print(df_with_targets[sample_cols].head(10))

# %%
# Analyze pump frequency by hour
print("Pump frequency by hour of day (15min targets):")
hourly_pumps = df_with_targets.groupby('hour')['target_pump_15min'].agg(['count', 'sum', 'mean'])
hourly_pumps['pump_rate'] = (hourly_pumps['mean'] * 100).round(2)
print(hourly_pumps)

# %%
# Check correlations between different time horizon targets
import numpy as np
target_pump_cols = [col for col in df_with_targets.columns if col.startswith('target_pump_') and col.endswith('min')]
correlation_matrix = df_with_targets[target_pump_cols].corr()
print("Correlation between different time horizon targets:")
print(correlation_matrix.round(3))

Missing Values:
620

Columns with missing values:
['SMA_20', 'SMA_50', 'SMA_200', 'RSI_14', 'RSI_21', 'BB_Upper', 'BB_Middle', 'BB_Lower', 'Stoch_K', 'Stoch_D', 'ATR', 'Wyckoff_Volume_ratio', 'Wyckoff_effort_result', 'Wyckoff_vpt', 'Wyckoff_vpt_ma', 'price_change', 'price_change_5min', 'volatility_10min', 'volatility_30min', 'Volume_change', 'Volume_ma_ratio', 'price_position_bb', 'momentum_5min', 'momentum_15min']
Price Statistics (Close):
count    36184.000000
mean      3555.298283
std        139.260758
min       3311.825000
25%       3404.738000
50%       3595.250000
75%       3657.945000
max       3790.155000
Name: Close, dtype: float64

Price Range: 3311.82 to 3790.16
Current price change std: 0.000225
Time Range:
From: 2025-08-20 00:00:00
To: 2025-09-25 07:36:00
Total records: 36184

Time frequency check (minutes):
Most common interval: -1.0
Unique intervals: [np.float64(-2941.0), np.float64(-212.0), np.float64(-61.0), np.float64(-3.0), np.float64(-1.0)]
Adding Target Columns for